In [1]:
import sys
sys.path.insert(0, '../lib')

import numpy as np
import pandas as pd

import common_data

In [2]:
%config InlineBackend.figure_format = "retina"

In [3]:
pd.options.display.max_columns = 300
pd.options.display.max_rows = 350
pd.options.display.max_colwidth = 10000

# Create supplementary table 37 with numbers of samples for all labels for all cohorts

- sample groups (broad in Fig 1b)
- pathogen groups
- VAP cure
- VAP onset
- virus detected
- virus only detected
- bacteria detected
- bacteria only detected

In [4]:
ehr = pd.read_csv(common_data.CLINICAL_LABELS, index_col=0)
ehr_data = pd.read_csv(common_data.CLINICAL, index_col=0)

In [5]:
ehr = ehr.merge(ehr_data, left_index=True, right_index=True, suffixes=('', '_ehr'))

In [6]:
METADATA_FIELDS = {
    'bal_barcode': 'bal_barcode',
    'perturbation_groups_2': 'Pathogen groups',
    'pathogen_groups': 'pathogen_groups',
    'is_culture_negative_pneumonia': 'Pathogen negative',
    'is_episode_cured': 'VAP cure in 7 days',
    'vap_onset_d7_empirical': 'VAP onset within 7 days',
    'Pathogen_bacteria_detected': 'Pathogen_bacteria_detected',
    'Pathogen_virus_detected': 'Pathogen_virus_detected',
    'episode_type': 'Pneumonia episode type',
    'patient': 'patient',
    'day_of_hospitalization': 'day_of_hospitalization',
    'Episode_duration': 'Episode_duration',
    'Episode_is_cured': 'Episode_is_cured',
    'ICU_stay': 'ICU_stay',
    'ICU_day': 'ICU_day',
    'Virus_cleared_on': 'Virus_cleared_on'
}

In [7]:
RARE_PATHOGENS = [
    'Gram-*; SARS-CoV-2',
    'Gram-*; SARS-CoV-2; Gram+',
    'Other viruses',
    'Gram-*; Pseudomonas aeruginosa',
    'Pseudomonas aeruginosa; Gram+',
    'Other viruses; Pseudomonas aeruginosa',
    'Gram-*; Other viruses; SARS-CoV-2'
]

In [8]:
labels = ehr[METADATA_FIELDS.keys()]

In [9]:
labels = labels.rename(columns=METADATA_FIELDS)

In [10]:
labels.loc[labels['Pathogen negative'].fillna(False), 'Sample group'] = 'Pathogen-negative pneumonia'

In [11]:
labels.loc[labels['pathogen_groups'].isin(RARE_PATHOGENS), 'Sample group'] = 'Rare pathogens'

In [12]:
labels['Pathogen groups'] = labels['Pathogen groups'].replace({
    'Pseudomonas aeruginosa': 'Pseudomonas',
    'Pseudomonas aeruginosa; SARS-CoV-2': 'SARS-CoV-2; Pseudomonas',
    'Gram-*': 'Other Gram–',
    'Gram-*; Gram+': 'Other Gram–; Gram+'
})

In [13]:
labels.loc[labels['Pathogen groups'].eq('NPC'), 'Sample group'] = 'NPC'
labels.loc[labels['Pathogen groups'].isin(['Early SARS-CoV-2', 'Late SARS-CoV-2']), 'Sample group'] = 'Viral'
labels.loc[labels['Pathogen groups'].isin(
    ['Pseudomonas', 'Gram+', 'Other Gram–', 'Other Gram–; Gram+']
), 'Sample group'] = 'Bacterial'
labels.loc[labels['Pathogen groups'].isin(
    ['SARS-CoV-2; Pseudomonas', 'Early SARS-CoV-2; Gram+', 'Late SARS-CoV-2; Gram+']
), 'Sample group'] = 'Mixed'
labels.loc[
    (
        labels['pathogen_groups'].eq('pathogen-negative')
        & labels['Sample group'].ne('Pathogen-negative pneumonia')
        & labels['Pathogen groups'].eq('discard')
        & labels['Pneumonia episode type'].isin(['CAP', 'HAP', 'VAP', 'VVAP'])
    ),
    'Sample group'
] = 'Pathogen cleared'
labels['Sample group'] = labels['Sample group'].fillna('Other')

In [14]:
labels['Sample group'].value_counts(dropna=False)

Other                          7918
Bacterial                      2271
Rare pathogens                 1214
NPC                            1127
Pathogen-negative pneumonia    1067
Viral                           816
Mixed                           601
Pathogen cleared                273
Name: Sample group, dtype: int64

In [15]:
labels['Virus_detected'] = labels.Pathogen_virus_detected.copy()
labels['Bacteria_detected'] = labels.Pathogen_bacteria_detected.copy()

In [16]:
idx = labels.Episode_duration.notna() & labels.Episode_is_cured.eq('Cured')
for i, row in labels.loc[idx].iterrows():
    try:
        target_idx = labels.index[int(labels.index.get_loc(i) + row.Episode_duration)]
    except:
        continue
    target_row = labels.loc[target_idx]
    if (row.patient == target_row.patient
            and row.ICU_stay == target_row.ICU_stay):
        if np.isnan(target_row.Virus_detected):
            labels.loc[target_idx, 'Virus_detected'] = False
        if np.isnan(target_row.Bacteria_detected):
            labels.loc[target_idx, 'Bacteria_detected'] = False

In [17]:
def ffill_detected_flags(patient_stay_df):
    result = []
    prev_vir = np.nan
    prev_bac = np.nan
    virus_cleared = None
    for _, row in patient_stay_df.iterrows():
        if not pd.isnull(row.Virus_cleared_on):
            virus_cleared = row.Virus_cleared_on
        # Clear viruses if virus_cleared
        if virus_cleared == row.ICU_day:
            prev_vir = False
        # Current BAL has priority
        if not pd.isnull(row.Virus_detected):
            prev_vir = row.Virus_detected
        if not pd.isnull(row.Bacteria_detected):
            prev_bac = row.Bacteria_detected
        result.append([prev_vir, prev_bac])
    return pd.DataFrame(result, index=patient_stay_df.index, columns=['Virus_detected', 'Bacteria_detected'])
# here groupby resorts rows, so we cannot just assign with `.values`
# we need to match by index
ffilled_pathogens = labels.groupby(['patient', 'ICU_stay'], group_keys=False).apply(ffill_detected_flags)
labels[['Virus_detected', 'Bacteria_detected']] = ffilled_pathogens

In [18]:
labels.Virus_detected = labels.Virus_detected.astype(float)
labels.Bacteria_detected = labels.Bacteria_detected.astype(float)

In [19]:
labels['Virus_exclusive'] = np.clip(labels.Virus_detected - labels.Bacteria_detected, 0, 1)

In [20]:
labels['Bacteria_exclusive'] = np.clip(labels.Bacteria_detected - labels.Virus_detected, 0, 1)

In [21]:
labels.Virus_detected.value_counts(dropna=False)

0.0    9292
1.0    3961
NaN    2034
Name: Virus_detected, dtype: int64

In [22]:
labels.Virus_exclusive.value_counts(dropna=False)

0.0    11078
1.0     2175
NaN     2034
Name: Virus_exclusive, dtype: int64

In [23]:
CATEGORIES = [
    'Sample group',
    'Pathogen groups',
    'VAP cure in 7 days',
    'VAP onset within 7 days',
    'Virus_detected',
    'Virus_exclusive',
    'Bacteria_detected',
    'Bacteria_exclusive'
]

In [24]:
PRETTY_CAT_NAMES_TXT = {
    'Virus_detected': 'Virus detected',
    'Virus_exclusive': 'Virus only detected',
    'Bacteria_detected': 'Bacteria detected',
    'Bacteria_exclusive': 'Bacteria only detected'
}

In [25]:
category_numbers = []
for column in CATEGORIES:
    column_vals = labels[column].replace({-1: np.nan}).dropna()
    for val in sorted(column_vals.unique()):
        count = labels[column].eq(val).sum()
        if column_vals.isin([0, 1]).all():
            val = bool(int(val))
        category_numbers.append(dict(
            category=PRETTY_CAT_NAMES_TXT.get(column, column).replace('_', ' ').replace('flag', '').strip(),
            value=val,
            count=count,
            cohort='EHR'
        ))
    if labels[column].replace({-1: np.nan}).isna().any():
        count = labels[column].replace({-1: np.nan}).isna().sum()
        category_numbers.append(dict(
            category=PRETTY_CAT_NAMES_TXT.get(column, column).replace('_', ' ').replace('flag', '').strip(),
            value='NA',
            count=count,
            cohort='EHR'
        ))
category_numbers = pd.DataFrame(category_numbers).rename(columns={
    'category': 'Variable',
    'value': 'Value',
    'count': 'Number of samples',
    'cohort': 'Cohort'
})

In [26]:
category_numbers.head()

,Variable,Value,Number of samples,Cohort
0,Sample group,Bacterial,2271,EHR
1,Sample group,Mixed,601,EHR
2,Sample group,NPC,1127,EHR
3,Sample group,Other,7918,EHR
4,Sample group,Pathogen cleared,273,EHR


In [27]:
all_categories = category_numbers.copy()

In [28]:
flow = pd.read_csv(common_data.RAW_FLOW, index_col=0)

In [29]:
flow.bal_barcode.isin(labels.bal_barcode).all()

True

In [30]:
flow_labels = labels.loc[labels.bal_barcode.isin(flow.bal_barcode)].copy()

In [31]:
flow_labels.shape

(792, 21)

In [32]:
category_numbers = []
for column in CATEGORIES:
    column_vals = flow_labels[column].replace({-1: np.nan}).dropna()
    for val in sorted(column_vals.unique()):
        count = flow_labels[column].eq(val).sum()
        if column_vals.isin([0, 1]).all():
            val = bool(int(val))
        category_numbers.append(dict(
            category=PRETTY_CAT_NAMES_TXT.get(column, column).replace('_', ' ').replace('flag', '').strip(),
            value=val,
            count=count,
            cohort='Flow cytometry'
        ))
    if flow_labels[column].replace({-1: np.nan}).isna().any():
        count = flow_labels[column].replace({-1: np.nan}).isna().sum()
        category_numbers.append(dict(
            category=PRETTY_CAT_NAMES_TXT.get(column, column).replace('_', ' ').replace('flag', '').strip(),
            value='NA',
            count=count,
            cohort='Flow cytometry'
        ))
category_numbers = pd.DataFrame(category_numbers).rename(columns={
    'category': 'Variable',
    'value': 'Value',
    'count': 'Number of samples',
    'cohort': 'Cohort'
})

In [33]:
all_categories = pd.concat([all_categories, category_numbers])

For scRNA-seq labels we cannot just repeat that, because we have Healthy controls, PASC and other samples

In [34]:
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)

In [35]:
sc_labels = sc_labels.merge(
    ehr_data,
    left_index=True,
    right_index=True,
    how='left',
    suffixes=('', '_ehr')
)

In [36]:
METADATA_FIELDS['cohort'] = 'cohort'

In [37]:
sc_labels = sc_labels[METADATA_FIELDS.keys()].set_index('bal_barcode')

In [38]:
sc_labels = sc_labels.rename(columns=METADATA_FIELDS)

In [39]:
sc_labels.loc[sc_labels['Pathogen negative'].fillna(False), 'Sample group'] = 'Pathogen-negative pneumonia'

In [40]:
sc_labels.loc[sc_labels['pathogen_groups'].isin(RARE_PATHOGENS), 'Sample group'] = 'Rare pathogens'

In [41]:
sc_labels['Pathogen groups'] = sc_labels['Pathogen groups'].replace({
    'Pseudomonas aeruginosa': 'Pseudomonas',
    'Pseudomonas aeruginosa; SARS-CoV-2': 'SARS-CoV-2; Pseudomonas',
    'Gram-*': 'Other Gram–',
    'Gram-*; Gram+': 'Other Gram–; Gram+'
})

In [42]:
sc_labels.loc[sc_labels['cohort'].eq('LongCOVID'), 'Sample group'] = 'PASC'
sc_labels.loc[sc_labels['Pathogen groups'].eq('Healthy'), 'Sample group'] = 'Healthy'
sc_labels.loc[sc_labels['Pathogen groups'].eq('NPC'), 'Sample group'] = 'NPC'
sc_labels.loc[sc_labels['Pathogen groups'].isin(['Early SARS-CoV-2', 'Late SARS-CoV-2']), 'Sample group'] = 'Viral'
sc_labels.loc[sc_labels['Pathogen groups'].isin(
    ['Pseudomonas', 'Gram+', 'Other Gram–', 'Other Gram–; Gram+']
), 'Sample group'] = 'Bacterial'
sc_labels.loc[sc_labels['Pathogen groups'].isin(
    ['SARS-CoV-2; Pseudomonas', 'Early SARS-CoV-2; Gram+', 'Late SARS-CoV-2; Gram+']
), 'Sample group'] = 'Mixed'
sc_labels.loc[
    (
        sc_labels['pathogen_groups'].eq('pathogen-negative')
        & sc_labels['Sample group'].ne('Pathogen-negative pneumonia')
        & sc_labels['Pathogen groups'].eq('discard')
        & sc_labels['Pneumonia episode type'].isin(['CAP', 'HAP', 'VAP', 'VVAP'])
    ),
    'Sample group'
] = 'Pathogen cleared'
sc_labels['Sample group'] = sc_labels['Sample group'].fillna('Other')

In [43]:
sc_labels['Sample group'].value_counts(dropna=False)

Viral                          60
Bacterial                      58
Mixed                          33
Pathogen-negative pneumonia    32
Other                          28
NPC                            26
PASC                           25
Rare pathogens                 22
Healthy                         9
Pathogen cleared                8
Name: Sample group, dtype: int64

In [44]:
sc_labels['Pathogen groups'] = sc_labels['Pathogen groups'].replace({'discard': np.nan})

In [46]:
sc_labels['Virus_detected'] = sc_labels.Pathogen_virus_detected.astype(float)
sc_labels['Bacteria_detected'] = sc_labels.Pathogen_bacteria_detected.astype(float)

In [47]:
sc_labels['Virus_exclusive'] = np.clip(sc_labels.Virus_detected - sc_labels.Bacteria_detected, 0, 1)

In [48]:
sc_labels['Bacteria_exclusive'] = np.clip(sc_labels.Bacteria_detected - sc_labels.Virus_detected, 0, 1)

In [49]:
category_numbers = []
for column in CATEGORIES:
    column_vals = sc_labels[column].replace({-1: np.nan}).dropna()
    for val in sorted(column_vals.unique()):
        count = sc_labels[column].eq(val).sum()
        if column_vals.isin([0, 1]).all():
            val = bool(int(val))
        category_numbers.append(dict(
            category=PRETTY_CAT_NAMES_TXT.get(column, column).replace('_', ' ').replace('flag', '').strip(),
            value=val,
            count=count,
            cohort='scRNA-seq'
        ))
    if sc_labels[column].replace({-1: np.nan}).isna().any():
        count = sc_labels[column].replace({-1: np.nan}).isna().sum()
        category_numbers.append(dict(
            category=PRETTY_CAT_NAMES_TXT.get(column, column).replace('_', ' ').replace('flag', '').strip(),
            value='NA',
            count=count,
            cohort='scRNA-seq'
        ))
category_numbers = pd.DataFrame(category_numbers).rename(columns={
    'category': 'Variable',
    'value': 'Value',
    'count': 'Number of samples',
    'cohort': 'Cohort'
})

In [51]:
all_categories = pd.concat([all_categories, category_numbers])

In [53]:
all_categories.to_csv('00_figures/15_cohort_label_numbers.csv')

In [54]:
all_categories

,Variable,Value,Number of samples,Cohort
0,Sample group,Bacterial,2271,EHR
1,Sample group,Mixed,601,EHR
2,Sample group,NPC,1127,EHR
3,Sample group,Other,7918,EHR
4,Sample group,Pathogen cleared,273,EHR
5,Sample group,Pathogen-negative pneumonia,1067,EHR
6,Sample group,Rare pathogens,1214,EHR
7,Sample group,Viral,816,EHR
8,Pathogen groups,Early SARS-CoV-2,119,EHR
9,Pathogen groups,Early SARS-CoV-2; Gram+,45,EHR
